# 04 · E1 baselines (Table 1)

**Runtime:** T4 GPU. **Time:** ~10 minutes.
Needs notebooks 01–03 (G1 passed).

SciQ, Pythia-160M, layer `L12`, mean pooling, full training data, evaluated on **dev** only
(the locked test set is not touched until E12).

| method | details |
|---|---|
| Zero-shot LM | log-prob of the option tokens given `Question: …\nAnswer:`; per-token normalized (primary), per-character and unnormalized also reported |
| Linear probe (map A) | s = wᵀh, softmax over the 4 options, L-BFGS, λ ∈ {1e-4 … 1e2}; deterministic, so one run |
| MLP head (map G) | 768 → 256 → GELU → dropout → 768 → score; AdamW lr 1e-3, wd 1e-2, batch 64, early stopping patience 10; dropout ∈ {0, 0.1, 0.3, 0.5} × 3 seeds |

Selection rule (fixed in advance): highest dev accuracy (mean over seeds for the MLP); ties go to
the stronger regularization. These are post-selection dev numbers, as the protocol intends.

**Gate:** if the linear probe is below 35 %, fix the representation before E2 (try `last` pooling and `L6`).

In [ ]:
# ---- Setup (edit the two lines below once) ------------------------------------------
GITHUB_REPO = "surensagh/PhotonicsAI-Project"                    # repo that holds this code
WORKSPACE   = "/content/drive/MyDrive/optical_lm_workspace"      # splits, cache, runs, reports
# -------------------------------------------------------------------------------------
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(WORKSPACE, exist_ok=True)
os.environ["OPTLM_WORKSPACE"] = WORKSPACE
os.environ["HF_HOME"] = "/content/hf_cache"          # downloads stay on the fast local disk

# Private repo: add a GitHub token as a Colab secret named GH_TOKEN (key icon in the left bar).
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
except Exception:
    token = None
url = f"https://{token}@github.com/{GITHUB_REPO}.git" if token else f"https://github.com/{GITHUB_REPO}.git"
REPO_DIR = "/content/" + GITHUB_REPO.split("/")[-1]
cmd = ["git", "clone", "-q", url, REPO_DIR] if not os.path.isdir(REPO_DIR) else ["git", "-C", REPO_DIR, "pull", "-q", "--ff-only"]
res = subprocess.run(cmd, capture_output=True, text=True)
if res.returncode != 0:
    msg = res.stderr.replace(token, "***") if token else res.stderr
    raise RuntimeError("git failed (check GITHUB_REPO / GH_TOKEN):\n" + msg)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("repo   :", REPO_DIR)
print("commit :", subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip())
print("dirty  :", bool(subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"]).strip()))
print("workspace:", WORKSPACE)

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
!python scripts/run_e1.py --config configs/e1.yaml

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(os.path.join(WORKSPACE, "reports", "e1_table1.md"), encoding="utf-8").read()))

Every run wrote a JSON record with config, commit hash, environment and per-item predictions to
`WORKSPACE/runs/E1/`. Write today's notebook entry from `notebook/TEMPLATE.md`.